# *Nextmv Hello World Example*

This notebook runs as a Databricks job and is triggered by a [workflow](https://www.nextmv.io/docs/using-nextmv/workflows/decision-workflows) on Nextmv.

It shows how you can run your model (or other code) on Databricks and view the results in the Nextmv UI. 

Why use Nextmv behind the scenes?
* View and share your results via the Nextmv UI
* Manage and compare multiple versions of your model
* Provide a no-code UI to others to test model parameters

## Prerequisites 
To run this notebook example you'll need to have your Nextmv API Key as a managed secret. 

 ```
      databricks secrets put-secret --json '{
        "scope": "<scope-name>",
        "key": "nextmv-api-key",
        "string_value": "<api-key-secret>"
```


In [0]:
### pip install your requirments

%pip install --upgrade "nextmv[all]"
%pip install plotly

In [0]:
%restart_python or dbutils.library.restartPython()

In [0]:
%python
api_key = dbutils.secrets.get(scope="my-scope", key="nextmv-api-key")

In [0]:
## Import libraries
import json

import nextmv
import plotly.graph_objects as go
from nextmv import cloud


In [0]:
### Using your api key, connect to Nextmv.

client = cloud.Client(api_key=api_key)

In [0]:
## Create an app space on Nextmv to store your results or use existing

app_name = "hello-world"
app = cloud.Application.new(client=client, id=app_name, name=app_name, exist_ok=True)

In [0]:
# Read the input from stdin. Note, if you collect data directly from other
# sources in your notebook that is fine, you can also read the data from
# a file or a database: input = nextmv.load_local(path="input.json")
input = {"name": "world", "radius": 6378, "distance": 147.6}

In [0]:
# Let's track a run to nextmv using the input data. We'll make our run more
# exciting later!

# This run is tracked in the hello-world app and the solution will be visible in
# the result tab.
result1 = app.track_run_with_result(
    tracked_run=cloud.TrackedRun(
        input=input,
        output=nextmv.Output(solution={"solution": [{"message": "Hello, world"}]}),
        status=cloud.TrackedRunStatus.SUCCEEDED,
        duration=10,
    )
)

print(f"Run {result1.id} tracked successfully.")

Now we can look at the run in our [Hello World app](https://cloud.nextmv.io/app/hello-world) and see the [Input](https://www.nextmv.io/docs/python-sdks/nextmv/input) and Result ([Output](https://www.nextmv.io/docs/python-sdks/nextmv/output)) for the run id above. Now let's do something with the input data and add more information via [Statistics](https://www.nextmv.io/docs/using-nextmv/reference/statistics) to our run.



In [0]:
# We can run our model and create nextmv.Output directly in the notebook.

##### Insert your more sophisticated model here

name = input["name"]
# Print logs that render in the run view in Nextmv Console.
message = f"Hello, {name}"

# Add statistics to your output.
output = nextmv.Output(
    solution={"solution": [{"message": message}]},
    statistics=nextmv.Statistics(
        result=nextmv.ResultStatistics(
            value=input["distance"],
            custom={"run_on_databricks": True},
        ),
    ),
)

In [0]:
# Create a visual for your data


def create_visuals(name: str, radius: float, distance: float) -> list[nextmv.Asset]:
    """Create a Plotly bar chart with radius and distance for a planet."""

    fig = go.Figure()
    fig.add_trace(
        go.Bar(x=[name], y=[radius], name="Radius (km)", marker_color="red", opacity=0.5),
    )
    fig.add_trace(
        go.Bar(
            x=[name],
            y=[distance],
            name="Distance (Millions km)",
            marker_color="blue",
            opacity=0.5,
        ),
    )
    fig.update_layout(
        title="Radius and Distance by Planet",
        xaxis_title="Planet",
        yaxis_title="Values",
        barmode="group",
    )
    fig = fig.to_json()

    ### Assets are used to render visualizations in Nextmv.
    assets = [
        nextmv.Asset(
            name="Plotly example",
            content_type="json",
            visual=nextmv.Visual(
                visual_schema=nextmv.VisualSchema.PLOTLY,
                visual_type="custom-tab",
                label="Charts",
            ),
            content=[json.loads(fig)],
        )
    ]
    return assets

In [0]:
# Add the visual to your output using assets.

assets = create_visuals(name, input["radius"], input["distance"])
output.assets = assets

In [0]:
# Track a new run with our statistics and visual assets.

result2 = app.track_run_with_result(
    tracked_run=cloud.TrackedRun(
        input=input,
        output=output,
        status=cloud.TrackedRunStatus.SUCCEEDED,
        duration=10,
    )
)

print(f"Run {result2.id} tracked successfully.")

Now we can look at the run in our [Hello World app](https://cloud.nextmv.io/app/hello-world) and see the Details (statistics) Input, Result (output), and Charts ([assets](https://www.nextmv.io/docs/using-nextmv/run/custom-visualization/overview)) for the new run id. 



In [0]:
# You can also add additional data to output from this notebook.

result2.output["statistics"]["result"]["custom"]["run_id"] = result2.id
result2.output["statistics"]["result"]["custom"]["app_id"] = app.id

In [0]:
# Output from the notebook that will surface in the workflow app on Nextmv

dbutils.notebook.exit(json.dumps(result2.output))

If we want to [push the model to Nextmv](https://www.nextmv.io/docs/python-sdks/nextmv/application/push) from this notebook, we'll need to create a [Nextmv Model](https://www.nextmv.io/docs/python-sdks/nextmv/model). This is nice if we want to manage multiple versions of the model and allow users to select which version to run in the UI. It also allows us to expose [options](https://www.nextmv.io/docs/python-sdks/nextmv/options) for scenario testing and other UI features.

In [0]:
### To push this model as a managed artifact that you can run on Nextmv, define the Model class.

### You can expose options to the UI and API for your model via Nextmv Options.
options = nextmv.Options(
    nextmv.Option("details", bool, True, "Print details to logs. Default true.", False),
)


class HelloWorld(nextmv.Model):
    def solve(self, input: nextmv.Input) -> nextmv.Output:
        ##### Insert your more sophisticated model here

        name = input.data["name"]
        # Print logs that render in the run view in Nextmv Console.
        message = f"Hello, {name}"
        nextmv.log(message)

        if options.details:
            detail = f"You are {input.data['distance']} million km from the sun"
            nextmv.log(detail)

        assets = create_visuals(name, input.data["radius"], input.data["distance"])

        # Output and statistics are shown in the Nextmv Console run view.
        output = nextmv.Output(
            solution={"solution": [{"message": message}]},
            statistics=nextmv.Statistics(
                result=nextmv.ResultStatistics(
                    value=1.23,
                    custom={"message": message},
                ),
            ),
            assets=assets,
        )
        return output


### Create visualizations for your model


def create_visuals(name: str, radius: float, distance: float) -> list[nextmv.Asset]:
    """Create a Plotly bar chart with radius and distance for a planet."""

    fig = go.Figure()
    fig.add_trace(
        go.Bar(x=[name], y=[radius], name="Radius (km)", marker_color="red", opacity=0.5),
    )
    fig.add_trace(
        go.Bar(x=[name], y=[distance], name="Distance (Millions km)", marker_color="blue", opacity=0.5),
    )
    fig.update_layout(
        title="Radius and Distance by Planet", xaxis_title="Planet", yaxis_title="Values", barmode="group"
    )
    fig = fig.to_json()

    ### Assets are used to render visualizations in Nextmv.
    assets = [
        nextmv.Asset(
            name="Plotly example",
            content_type="json",
            visual=nextmv.Visual(
                visual_schema=nextmv.VisualSchema.PLOTLY,
                visual_type="custom-tab",
                label="Charts",
            ),
            content=[json.loads(fig)],
        )
    ]

    return assets

In [0]:
### Solve the hello world model on your databricks compute (local).
model = HelloWorld()
output = model.solve(input)

### Write the output to a file and print.
nextmv.write_local(output, path="output.json")

In [0]:
# Optional! Push a model to run on Nextmv and share the model with others via the UI

model_configuration = nextmv.ModelConfiguration(
    name="hello-world",
    requirements=["nextmv==1.1.2", "plotly==6.0.0"],
    options=options,
)
manifest = nextmv.cloud.Manifest.from_model_configuration(model_configuration)

app.push(
    manifest=manifest,
    model=model,
    model_configuration=model_configuration,
    verbose=True,
)

In [0]:
### Create a new run from the notebook that executes on Nextmv compute using the hello-world app

app_name = "hello-world"
app = cloud.Application(client=client, id=app_name)
result = app.new_run(input=input.data)

Visit the `Apps + Workflows` space on [https://cloud.nextmv.io/apps](https://cloud.nextmv.io/apps) to view results in your `Hello World` app.